# 04 — Does the order of the tracks matter?

**Question:** does the probability of skipping a track depend on the track before it, and not only on whether that one was skipped? If it does, ordering the playlist is an optimization problem on the arcs (orienteering-like); if it does not, the order has no value.

**Rows:** as in the model (`model_rows`), with a known previous track in the session; test from 2025-07-01, with a logistic regression trained before the cut to compare predicted and actual rates.

In [1]:
import sys

import joblib
import numpy as np
import pandas as pd
from scipy.special import logit

sys.path.append("../src")
from features import ROOT
from playlist import F, PARAMS, solve_v1
from train import MODEL_PATH, candidate_features, model_rows, predict_candidates, to_Z, train

df = pd.read_parquet(ROOT / "data" / "features.parquet")

# Is the previous track of the session by the same artist?
prev_artist = df.groupby("session_id")["artist"].shift(1)
df["same_artist"] = df["artist"] == prev_artist

rows = model_rows(df)
CUT = pd.Timestamp("2025-07-01", tz="UTC")
tr = rows[rows["ts"] < CUT]
te = rows[(rows["ts"] >= CUT) & rows["prev_y"].notna()].copy()
te["p"] = train(tr).predict_proba(to_Z(te))[:, 1]


def compare(d, by):
    """Number of rows, actual and predicted skip rate for each group."""
    return d.groupby(by, observed=True).agg(
        n=("y", "size"), actual=("y", "mean"), predicted=("p", "mean")
    ).round(3)

## Same artist as the previous track

In [2]:
compare(te, ["shuffle", "prev_y", "same_artist"])

n  actual  predicted
shuffle prev_y same_artist                         
False   0.0    False        2117   0.063      0.067
               True          914   0.063      0.080
        1.0    False         339   0.619      0.563
               True           91   0.473      0.586
True    0.0    False        9137   0.131      0.108
               True         2911   0.057      0.072
        1.0    False        3295   0.599      0.607
               True          394   0.556      0.599

In [3]:
# Share of same-artist pairs in the session, the row itself excluded
g = df.groupby("session_id")["same_artist"]
df["sess_share"] = (g.transform("sum") - df["same_artist"]) / (g.transform("size") - 1)

te = te.join(df["sess_share"])
te["variety"] = pd.cut(te["sess_share"], [-0.01, 0.05, 0.15, 0.4, 1.0])

# In shuffle, previous track not skipped
x = te[te["shuffle"] & (te["prev_y"] == 0)]
compare(x, ["variety", "same_artist"])

n  actual  predicted
variety       same_artist                         
(-0.01, 0.05] False        3522   0.141      0.109
              True           98   0.133      0.091
(0.05, 0.15]  False        2545   0.138      0.113
              True          151   0.113      0.096
(0.15, 0.4]   False        2300   0.119      0.106
              True          783   0.074      0.080
(0.4, 1.0]    False         770   0.101      0.090
              True         1879   0.042      0.066

## Similarity from co-occurrence

Two tracks are similar if they appear in the same sessions (sessions of the training period, cosine).

In [4]:
past = df[df["ts"] < CUT]
sessions_of = past.groupby("track_uri")["session_id"].agg(set)

df["prev_uri"] = df.groupby("session_id")["track_uri"].shift(1)
te = te.join(df["prev_uri"])


def cooc(a, b):
    """Similarity of two tracks: sessions in common in the training period, normalized (cosine)."""
    if a not in sessions_of.index or b not in sessions_of.index:
        return np.nan
    sa, sb = sessions_of[a], sessions_of[b]
    return len(sa & sb) / np.sqrt(len(sa) * len(sb))


te["sim"] = [cooc(a, b) for a, b in zip(te["prev_uri"], te["track_uri"])]
te["sim_bin"] = pd.cut(te["sim"], [-0.01, 0, 0.05, 0.15, 1.0])

x = te[te["shuffle"] & (te["prev_y"] == 0)]
print("pairs without a similarity (new tracks):", round(x["sim"].isna().mean(), 3))
print(compare(x, "sim_bin"), "\n")

# Varied sessions only (at most 15% of same-artist pairs), where the order is random
compare(x[x["sess_share"] <= 0.15], ["same_artist", "sim_bin"])

pairs without a similarity (new tracks): 0.457
                 n  actual  predicted
sim_bin                              
(-0.01, 0.0]  4517   0.156      0.117
(0.0, 0.05]    535   0.144      0.116
(0.05, 0.15]   875   0.150      0.108
(0.15, 1.0]    614   0.096      0.081 



n  actual  predicted
same_artist sim_bin                              
False       (-0.01, 0.0]  2806   0.179      0.123
            (0.0, 0.05]    299   0.164      0.127
            (0.05, 0.15]   389   0.177      0.118
            (0.15, 1.0]     68   0.103      0.108
True        (-0.01, 0.0]    53   0.113      0.087
            (0.0, 0.05]     14   0.071      0.089
            (0.05, 0.15]    25   0.200      0.095
            (0.15, 1.0]     32   0.219      0.098

## How much can the order be worth?

Simulation on the tracks of version 1 (same moment as notebook 03, final model). Without pair effects the skip probability of the $k$-th track depends only on the outcome of the previous one: $P_k = p0_k + (p1_k - p0_k)\,P_{k-1}$, and the expected number of skips is $\sum_k P_k$.

In [5]:
streams = pd.read_parquet(ROOT / "data" / "streams_clean.parquet")
pool = pd.read_parquet(ROOT / "data" / "pool.parquet")
model = joblib.load(MODEL_PATH)

context = dict(platform="ios", country="DE", shuffle=True)
feats = candidate_features(streams, pool["track_uri"], "2026-10-26 08:00", **context)
cand = predict_candidates(model, feats)
cand = cand.merge(pool[["track_uri", "duration_s", "is_new"]], on="track_uri")
cand["d"] = cand["duration_s"] / 60
cand["v"] = (1 - (1 - F) * cand["p0"]) * cand["d"]
pl = cand[solve_v1(cand, **PARAMS)["x"] > 0.5]

# In the logistic regression prev_y is an additive term without interactions:
# logit(p1) - logit(p0) is the same for every track, so p1 is a function of p0
shift = logit(cand["p1"]) - logit(cand["p0"])
print(f"logit(p1) - logit(p0) on the candidates: mean {shift.mean():.3f}, std {shift.std():.1e}")

logit(p1) - logit(p0) on the candidates: mean 2.463, std 2.8e-16


In [6]:
def expected_skips(p0, p1):
    """Expected skips in a sequence; the first track comes after a non-skipped one."""
    P, total = 0.0, 0.0
    for a, b in zip(p0, p1):
        P = a + (b - a) * P
        total += P
    return total


rng = np.random.default_rng(0)
p0, p1 = pl["p0"].to_numpy(), pl["p1"].to_numpy()
orders = [rng.permutation(len(pl)) for _ in range(20_000)]
sims = np.array([expected_skips(p0[o], p1[o]) for o in orders])
print(
    f"{len(pl)} tracks, 20,000 random orders, expected skips: "
    f"best {sims.min():.3f}, worst {sims.max():.3f}"
)

# Hypothetical case: p1 very different between tracks (between 0.2 and 0.7), unlike the model
p1_h = rng.uniform(0.2, 0.7, len(pl))
sims_h = np.array([expected_skips(p0[o], p1_h[o]) for o in orders])
print(f"with p1 between 0.2 and 0.7: best {sims_h.min():.3f}, worst {sims_h.max():.3f}")

# Independence assumption of version 1: every track as if it came after a non-skip
print(
    f"\nexpected skips, independence (sum of p0): {p0.sum():.3f}; "
    f"chain (mean over the orders): {sims.mean():.3f}"
)

15 tracks, 20,000 random orders, expected skips: best 0.853, worst 0.857
with p1 between 0.2 and 0.7: best 0.929, worst 1.060

expected skips, independence (sum of p0): 0.621; chain (mean over the orders): 0.855


## Conclusions

- **Same artist as the previous track.** In shuffle after a non-skipped track: 13.1% of skips when the artist changes (n = 9,137), 5.7% when it is the same (n = 2,911); the model already predicts about half of the gap (10.8% against 7.2%). Without shuffle no difference (6.3% in both cases).
- **It is the context, not the pair.** Same-artist pairs are 21% of the pairs in shuffle, too many for a large library shuffled at random: shuffle on an artist page, an album, a small playlist. In varied sessions (at most 5% of same-artist pairs), where such a pair happens by chance, the rates are 14.1% and 13.3% (n = 98); the gap appears only in sessions concentrated on one artist (above 40%: 10.1% against 4.2%). It is the intention of listening to that artist, and putting two of its tracks side by side in a playlist does not recreate it.
- **Method.** Within a shuffle session over a varied context Spotify decides the order at random: a natural experiment that measures pair effects without confounding.
- **Similarity from co-occurrence** (not computable for 46% of the pairs, new tracks): in shuffle after a non-skip the rate is flat between similarity 0 and 0.15 (15.6, 14.4, 15.0%) and drops to 9.6% above 0.15, where the model already predicts 8.1%. In the varied sessions the only low group has n = 68 (±7 points).
- **Value of the order.** In the model `p1` is a function of `p0` (the logit shift is the same for every track, 2.46): both are properties of the current track, not of the pair. The problem is a sequencing with costs on the nodes, not a TSP or orienteering with costs on the arcs. On the 15 tracks of version 1, the best and the worst of 20,000 orders differ by 0.004 expected skips (0.855 against 0.859). Even with a `p1` between 0.2 and 0.7 changing from track to track the range is 0.13 skips, under one point of skip rate: not measurable.
- **Decision:** no pair effect where the order is random, and no value in the order with the model: version 2 "order" is not developed.
- **Side effect:** the independence assumption of version 1 (every track as if after a non-skip) underestimates the expected skips by about a quarter (0.62 against 0.86 over 15 tracks). It is fine for choosing the tracks, but the absolute expected minutes are optimistic: to be stated in the README.